In [33]:
%iam_role arn:aws:iam::770170581396:role/aws-glue-s3-permission
%region eu-north-1
%idle_timeout 15
%worker_type G.1X
%number_of_workers 2
%glue_version 4.0

You are already connected to a glueetl session 3b24f15c-7045-4bc8-a3e9-7979edb4e7e4.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Current iam_role is arn:aws:iam::770170581396:role/aws-glue-s3-permission
iam_role has been set to arn:aws:iam::770170581396:role/aws-glue-s3-permission.


You are already connected to a glueetl session 3b24f15c-7045-4bc8-a3e9-7979edb4e7e4.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous region: eu-north-1
Setting new region to: eu-north-1
Region is set to: eu-north-1


You are already connected to a glueetl session 3b24f15c-7045-4bc8-a3e9-7979edb4e7e4.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


...

Previous worker type: G.1X
Setting new worker type to: G.1X


You are already connected to a glueetl session 3b24f15c-7045-4bc8-a3e9-7979edb4e7e4.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous number of workers: 2
Setting new number of workers to: 2


You are already connected to a glueetl session 3b24f15c-7045-4bc8-a3e9-7979edb4e7e4.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Setting Glue version to: 4.0


# Bronze Layer — Development Notebook

**Dataset**: Amazon Fine Food Reviews — 10-column CSV uploaded to S3 raw folder

**Columns**: `Id`, `ProductId`, `UserId`, `ProfileName`, `HelpfulnessNumerator`, `HelpfulnessDenominator`, `Score`, `Time`, `Summary`, `Text`

**Steps**:
1. Read raw CSV from `s3://amazon-food-reviews-ml-model/raw/`
2. EDA — schema, nulls, Score distribution, text length, duplicates
3. Bronze transformation — rename, cast, derive label, add metadata

## 1. Read Raw CSV from S3

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType

# Full CSV already uploaded to S3 raw folder
RAW_PATH = "s3://amazon-food-reviews-ml-model/raw/Reviews.csv"

# multiLine=True  -> handles review Text with embedded newlines
# escape='"'      -> correctly handles double-quoted fields
raw_df = (
    spark.read
    .option("header",      True)
    .option("inferSchema", True)
    .option("multiLine",   True)
    .option("escape",      '"')
    .option("quote",       '"')
    .csv(RAW_PATH)
)

raw_df.printSchema()
print(f"\nTotal rows: {raw_df.count()}")
raw_df.show(5, truncate=80)

root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = true)
 |-- HelpfulnessNumerator: integer (nullable = true)
 |-- HelpfulnessDenominator: integer (nullable = true)
 |-- Score: integer (nullable = true)
 |-- Time: integer (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)


Total rows: 568454
+---+----------+--------------+-------------------------------+--------------------+----------------------+-----+----------+---------------------+--------------------------------------------------------------------------------+
| Id| ProductId|        UserId|                    ProfileName|HelpfulnessNumerator|HelpfulnessDenominator|Score|      Time|              Summary|                                                                            Text|
+---+----------+--------------+-------------------------------+--------------------+-------------------

## 2. Exploratory Data Analysis (EDA)

In [2]:
# ── 2a. Shape & descriptive statistics ──────────────────────────────────────
print("=" * 60)
print("SHAPE")
print("=" * 60)
print(f"  Rows    : {raw_df.count()}")
print(f"  Columns : {len(raw_df.columns)}")
print(f"  Names   : {raw_df.columns}")

print()
print("=" * 60)
print("DESCRIPTIVE STATS")
print("=" * 60)
raw_df.select(
    "HelpfulnessNumerator",
    "HelpfulnessDenominator",
    "Score"
).describe().show()

SHAPE
  Rows    : 568454
  Columns : 10
  Names   : ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']

DESCRIPTIVE STATS
+-------+--------------------+----------------------+------------------+
|summary|HelpfulnessNumerator|HelpfulnessDenominator|             Score|
+-------+--------------------+----------------------+------------------+
|  count|              568454|                568454|            568454|
|   mean|  1.7438174416927315|    2.2288100708236724| 4.183198640523243|
| stddev|   7.636512706820835|     8.289740293185577|1.3104360248243097|
|    min|                   0|                     0|                 1|
|    max|                 866|                   923|                 5|
+-------+--------------------+----------------------+------------------+


In [3]:
# ── 2b. Null / missing value count per column ────────────────────────────────
print("=" * 60)
print("NULL COUNTS PER COLUMN")
print("=" * 60)
raw_df.select([
    F.count(F.when(
        F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
        c
    )).alias(c)
    for c in raw_df.columns
]).show()

NULL COUNTS PER COLUMN
+---+---------+------+-----------+--------------------+----------------------+-----+----+-------+----+
| Id|ProductId|UserId|ProfileName|HelpfulnessNumerator|HelpfulnessDenominator|Score|Time|Summary|Text|
+---+---------+------+-----------+--------------------+----------------------+-----+----+-------+----+
|  0|        0|     0|          0|                   0|                     0|    0|   0|      0|   0|
+---+---------+------+-----------+--------------------+----------------------+-----+----+-------+----+


In [4]:
# ── 2c. Score distribution ──────────────────────────────────────────────────
print("=" * 60)
print("SCORE DISTRIBUTION  (1=worst  5=best)")
print("=" * 60)
raw_df.groupBy("Score").count().orderBy("Score").show()

# Preview the binary label we will derive
print("=" * 60)
print("BINARY LABEL PREVIEW  (Score >= 4 -> Positive=1, else 0)")
print("=" * 60)
raw_df.withColumn(
    "Positive",
    F.when(F.col("Score") >= 4, 1).otherwise(0)
).groupBy("Positive").count().orderBy("Positive").show()

SCORE DISTRIBUTION  (1=worst  5=best)
+-----+------+
|Score| count|
+-----+------+
|    1| 52268|
|    2| 29769|
|    3| 42640|
|    4| 80655|
|    5|363122|
+-----+------+

BINARY LABEL PREVIEW  (Score >= 4 -> Positive=1, else 0)
+--------+------+
|Positive| count|
+--------+------+
|       0|124677|
|       1|443777|
+--------+------+


In [3]:
# ── 2d. Review text length analysis ────────────────────────────────────────
print("=" * 60)
print("REVIEW TEXT LENGTH STATS (characters)")
print("=" * 60)
raw_df.withColumn("text_len", F.length(F.col("Text"))).select(
    F.min("text_len").alias("min_chars"),
    F.round(F.avg("text_len"), 1).alias("avg_chars"),
    F.percentile_approx("text_len", 0.5).alias("median_chars"),
    F.max("text_len").alias("max_chars"),
).show()

# Suspiciously short reviews (< 10 chars)
short = raw_df.filter(F.length(F.col("Text")) < 10)
print(f"Reviews with < 10 characters: {short.count()}")
short.select("Id", "Score", "Text").show(10, truncate=False)

REVIEW TEXT LENGTH STATS (characters)
+---------+---------+------------+---------+
|min_chars|avg_chars|median_chars|max_chars|
+---------+---------+------------+---------+
|       12|    436.2|         302|    21409|
+---------+---------+------------+---------+

Reviews with < 10 characters: 0
+---+-----+----+
|Id |Score|Text|
+---+-----+----+
+---+-----+----+


In [4]:
# ── 2e. Duplicate detection ─────────────────────────────────────────────────
total    = raw_df.count()
distinct = raw_df.dropDuplicates(["Text"]).count()
print(f"Total rows            : {total}")
print(f"Distinct review texts : {distinct}")
print(f"Duplicate rows        : {total - distinct}")

Total rows            : 568454
Distinct review texts : 393579
Duplicate rows        : 174875


## 3. Bronze Layer Transformation

Minimal, traceable cleaning applied to the raw data:

| Step | Detail |
|------|--------|
| **Preserve** | Keep all 10 source fields using snake_case names |
| **Cast** | IDs/time → long · helpfulness/score → int |
| **Time safety** | Preserve `review_time_raw`; invalid epoch values become null |
| **Validate** | Require ID, usable text, score 1–5, and valid helpfulness counts |
| **Derive label** | `positive = 1` if `score >= 4`, else `0` |
| **Metadata** | `_ingested_at` · `_source_file` · deterministic `_record_id` |

In [ ]:
time_raw = F.trim(F.col("Time").cast("string"))
review_time_epoch = time_raw.cast("long")

bronze_df = (
    raw_df
    .select(
        F.col("Id").cast("long").alias("id"),
        F.trim(F.col("ProductId")).alias("product_id"),
        F.trim(F.col("UserId")).alias("user_id"),
        F.trim(F.col("ProfileName")).alias("profile_name"),
        F.col("HelpfulnessNumerator").cast("int").alias("helpfulness_numerator"),
        F.col("HelpfulnessDenominator").cast("int").alias("helpfulness_denominator"),
        F.col("Score").cast("int").alias("score"),
        time_raw.alias("review_time_raw"),
        review_time_epoch.alias("review_time_epoch"),
        F.from_unixtime(review_time_epoch).cast("timestamp").alias("review_timestamp"),
        F.trim(F.col("Summary")).alias("summary"),
        F.trim(F.col("Text")).alias("review_text"),
        F.input_file_name().alias("_source_file"),
    )
    .withColumn(
        "is_valid_record",
        F.col("id").isNotNull()
        & F.col("score").between(1, 5)
        & F.col("review_text").isNotNull()
        & (F.length(F.col("review_text")) > 0)
        & F.col("helpfulness_numerator").isNotNull()
        & F.col("helpfulness_denominator").isNotNull()
        & (F.col("helpfulness_numerator") <= F.col("helpfulness_denominator")),
    )
    .filter(F.col("is_valid_record"))
    .withColumn("positive", F.when(F.col("score") >= 4, 1).otherwise(0).cast("int"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn(
        "_record_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("_source_file"), F.lit("")),
                F.coalesce(F.col("id").cast("string"), F.lit("")),
                F.coalesce(F.col("product_id"), F.lit("")),
                F.coalesce(F.col("user_id"), F.lit("")),
            ),
            256,
        ),
    )
)

bronze_df.printSchema()
print(f"\nBronze row count: {bronze_df.count()}")
bronze_df.show(5, truncate=80)

In [ ]:
# ── Verify bronze output ────────────────────────────────────────────────────
print("Label distribution:")
bronze_df.groupBy("positive").count().orderBy("positive").show()

print("Score vs Positive label (sanity check):")
bronze_df.groupBy("score", "positive").count().orderBy("score").show()

print("Null check on bronze columns:")
bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["id", "score", "review_text", "positive", "_ingested_at", "_record_id"]
]).show()

## 4. Save Bronze Layer as Parquet

In [ ]:
BRONZE_PATH = "s3://amazon-food-reviews-ml-model/bronze/"

bronze_count = bronze_df.count()
if bronze_count == 0:
    raise ValueError("Bronze transformation produced zero valid records")

(
    bronze_df.write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(BRONZE_PATH)
)

# Verify the Parquet output after writing it.
written_bronze_df = spark.read.parquet(BRONZE_PATH)
written_count = written_bronze_df.count()

if written_count != bronze_count:
    raise ValueError(
        f"Bronze write verification failed: expected {bronze_count}, got {written_count}"
    )

print(f"Saved {written_count} Bronze rows as Parquet to:")
print(BRONZE_PATH)
written_bronze_df.printSchema()

## 5. Create a Validated 100-Row Sample

Select exactly 20 valid reviews from each score (1–5) and save one CSV object to S3.

In [5]:
from pyspark.sql import Window
import boto3
import csv
import io

SAMPLE_BUCKET = "amazon-food-reviews-ml-model"
SAMPLE_KEY = "sample-dataset/Reviews_sample_100.csv"

# Reject shifted/malformed CSV records before sampling. Casting invalid values
# to a numeric type produces null, so those records do not pass these checks.
valid_reviews_df = raw_df.filter(
    F.col("Id").cast("long").isNotNull()
    & F.col("HelpfulnessNumerator").cast("int").isNotNull()
    & F.col("HelpfulnessDenominator").cast("int").isNotNull()
    & F.col("Score").cast("int").between(1, 5)
    & F.col("Time").cast("long").isNotNull()
    & F.col("Text").isNotNull()
    & (F.length(F.trim(F.col("Text"))) > 0)
    & (
        F.col("HelpfulnessNumerator").cast("int")
        <= F.col("HelpfulnessDenominator").cast("int")
    )
)

# Stop instead of silently producing an incomplete or corrupt sample.
score_counts = {
    int(row["Score"]): row["count"]
    for row in valid_reviews_df.groupBy("Score").count().collect()
}
missing_scores = {score: score_counts.get(score, 0) for score in range(1, 6) if score_counts.get(score, 0) < 20}
if missing_scores:
    raise ValueError(f"Not enough valid reviews for balanced sampling: {missing_scores}")

score_window = Window.partitionBy("Score").orderBy(F.rand(seed=42))
sample_100_df = (
    valid_reviews_df
    .withColumn("_sample_row", F.row_number().over(score_window))
    .filter(F.col("_sample_row") <= 20)
    .drop("_sample_row")
    .orderBy(F.rand(seed=42))
)

sample_rows = sample_100_df.collect()
if len(sample_rows) != 100:
    raise ValueError(f"Expected 100 sampled rows, got {len(sample_rows)}")

# Create one correctly quoted CSV file and upload it as one S3 object.
csv_buffer = io.StringIO(newline="")
writer = csv.DictWriter(
    csv_buffer,
    fieldnames=sample_100_df.columns,
    quoting=csv.QUOTE_MINIMAL,
    lineterminator="\n",
)
writer.writeheader()
writer.writerows(row.asDict(recursive=True) for row in sample_rows)

csv_bytes = csv_buffer.getvalue().encode("utf-8")
s3 = boto3.client("s3", region_name="eu-north-1")
s3.put_object(
    Bucket=SAMPLE_BUCKET,
    Key=SAMPLE_KEY,
    Body=csv_bytes,
    ContentType="text/csv",
)

# Verify both the sample balance and the uploaded S3 object.
sample_100_df.groupBy("Score").count().orderBy("Score").show()
uploaded = s3.head_object(Bucket=SAMPLE_BUCKET, Key=SAMPLE_KEY)
print(f"Saved {len(sample_rows)} valid rows ({uploaded['ContentLength']} bytes) to:")
print(f"s3://{SAMPLE_BUCKET}/{SAMPLE_KEY}")

+-----+-----+
|Score|count|
+-----+-----+
|    1|   20|
|    2|   20|
|    3|   20|
|    4|   20|
|    5|   20|
+-----+-----+

Saved 100 valid rows (61980 bytes) to:
s3://amazon-food-reviews-ml-model/sample-dataset/Reviews_sample_100.csv


In [ ]:
%stop_session